# Week 1 — Topic 1: Python basics for data work

Welcome. This notebook is written for you if you are **new to coding** or new to Python. You are not expected to know jargon yet — we introduce words when we need them.

### How to use this notebook

- Run cells **from top to bottom** the first time (each section builds on the last).
- In Jupyter, click a cell and press **Run** (or use the run shortcut). 
- In many notebooks, **the last line of a code cell is shown as the “result”** below the cell. That is normal.
- If something fails, read the red error text once — it usually points to a line number. You can ask your tutor what it means.

### What “data engineering” means here (one picture)

Later you will move large amounts of data between systems. **Very often the first step is small:** one row from a file looks like a **record** (for example one product, one order, one customer). Your job in this topic is to learn how to **clean and transform one record safely** with Python — that skill scales to huge files later.

**Rule for this notebook:** use only what comes with Python (the **standard library**). No extra packages yet.

---

## Part 1 — Words you will see in this notebook

| Word | Plain meaning |
|------|----------------|
| **Python** | The programming language we use to give precise instructions to the computer. |
| **String** | Text in quotes, like `"hello"`. |
| **`None`** | A special value that means “there is no value here” (similar to empty in a form). |
| **Dictionary (`dict`)** | A bag of **labeled** values: each label is a **key**, each value is the **value**. Like a paper form with field names. |
| **Function** | A named mini-program: you give it inputs, it returns a result. |
| **Return** | The function hands a value back to you. |
| **Mutate / in place** | Change an object directly. We will often **avoid** mutating the input row so we still have the original for debugging. |

You do not need to memorize this table. Come back to it when a word feels fuzzy.

## Part 2 — Cleaning text: `strip`

Real data often has **extra spaces** around text: `"  ABC  "` instead of `"ABC"`. That can break joins (“ABC” does not equal ” ABC ”).

Python strings have a method called **`.strip()`** that removes spaces, tabs, and newlines **from the start and end only** (not from the middle).

**Important detail:** `strip` does **not** change the original string. It **returns a new** string. That is why we write `cleaned = raw.strip()` — we store the new version in `cleaned`.

Run the cell below. Then change the text inside `raw` and run again (for example add more spaces).

In [ ]:
raw = "  SKU-99  "
cleaned = raw.strip()

# Original is still the old value:
print("raw:", repr(raw))
print("cleaned:", repr(cleaned))

## Part 3 — “Missing” values: `None`, empty string, spaces only

When a CSV cell is empty, you might see:

- **`None`** in Python (similar to JSON `null`), or
- an **empty string** `""`, or
- a string that is **only spaces** `"   "`.

For **text fields** (name, description), many pipelines treat all of those as **“missing”** and store **`None`** in the cleaned row. That way “no description” is always represented the **same** way, which makes later steps easier.

Read the `is_missing_text` function below slowly:

1. If the value is `None`, it is missing.
2. If it is a string, we **strip** it; if nothing is left, it is missing.
3. Otherwise it is not missing.

Run the cell and look at the four `True`/`False` results.

In [ ]:
def is_missing_text(value):
    if value is None:
        return True
    if isinstance(value, str) and value.strip() == "":
        return True
    return False

print("None ->", is_missing_text(None))
print("empty string ->", is_missing_text(""))
print("only spaces ->", is_missing_text("   "))
print("normal text ->", is_missing_text("ok"))

## Part 4 — One row as a `dict`

A single row is often stored as a **dictionary**:

```text
key (field name)  ->  value
"sku"             ->  "A1"
"name"            ->  "Widget"
```

In Python you write that as `{"sku": "A1", "name": "Widget"}`.

Below we build a **new** dictionary `out` by looping over each **key** and **value** in `row`. For string values we **strip** spaces. This pattern is the heart of “clean one row.”

In [ ]:
row = {"sku": "  A1 ", "note": "x"}
out = {}

for key, val in row.items():
    if isinstance(val, str):
        out[key] = val.strip()
    else:
        out[key] = val

print("original row:", row)
print("cleaned copy:", out)

## Part 5 — From “blank string” to `None`

Stripping turns `"   "` into `""`, but we still have an empty string. For optional text fields we often want **`None`** instead.

The helper below:

- returns **`None`** for `None` or blank text after strip,
- returns the **stripped** string if there is real text,
- leaves non-strings as they are (simple rule for this lesson).

Run it and read the printed results.

In [ ]:
def optional_text(value):
    if value is None:
        return None
    if not isinstance(value, str):
        return value
    s = value.strip()
    return None if s == "" else s

print(optional_text("   "))   # should be None
print(optional_text("hello"))  # should be hello
print(optional_text(None))     # should be None

## Part 6 — Quick self-check (no code — think or say out loud)

1. Why do we use `None` instead of `""` for “missing name” in many pipelines?
2. Why is `cleaned = raw.strip()` safer than changing `raw` in place?
3. If you forget and edit `row` directly, what could you lose?

Ask your tutor for feedback on your answers when you are ready.

---

## Challenge A — `clean_row(row)`

**Goal:** Write a function that takes **one** dictionary (one row), returns a **new** dictionary, and:

- For **every string value**: strip whitespace. If the value is **missing** (`None`, `''`, or only spaces after strip), put **`None`** in the output.
- For **non-string** values: keep them as they are in this lesson (CSV numbers are still often strings — that is OK for now).
- **Do not** change the input `row`.

**Example:**  
`{"sku": "  A1 ", "name": "", "qty": " 5 "}` → `{"sku": "A1", "name": None, "qty": "5"}`  
Notice `qty` stays a **string** `"5"` unless you choose to convert it — that is fine here.

### Hints (use only what you need)

1. Create an empty dict like `out = {}`.
2. Loop with `for key, val in row.items():`.
3. If `val` is a string, you can reuse the idea from `optional_text`, or combine strip + checks.
4. After your function works, add an **`assert`** line to check the example above. `assert` means “Python should stop if this is not true.”

In [ ]:
def clean_row(row: dict) -> dict:
    raise NotImplementedError("Your turn: delete this line and write your code.")

# When your function is ready, uncomment:
# assert clean_row({"sku": "  A1 ", "name": "", "qty": " 5 "}) == {"sku": "A1", "name": None, "qty": "5"}
# print("Challenge A: OK")

---

## Challenge B — A tiny “price rule”

**Why this matters:** later you will **validate** data before loading it. Instead of crashing, you often return **“valid or not”** plus a **short reason**.

We will return **two values** packed together — a **tuple**:

- first value: **`True`** if the price is OK, **`False`** if not,
- second value: **`None`** if OK, or a **string code** like `"missing_price"` if not OK.

Example idea (not the full solution): `(False, "missing_price")` means “not OK, because price is missing.”

### Try this first — turning text into a number

`float("12.5")` converts the string `"12.5"` to a number Python can compare. If the text is not a number, Python **raises** an error — in Challenge B you will **catch** that with `try` / `except` (your tutor can show this the first time).

Run the cell below.

In [ ]:
print(float("19.99"))

try:
    float("not-a-number")
except ValueError as e:
    print("Could not convert — that is expected:", e)

### Your task — `apply_price_rule(price_str)`

Define **`apply_price_rule(price_str)`** so it returns a tuple according to these rules:

1. If `price_str` is missing or `""` or only spaces after strip → `(False, "missing_price")`.
2. Else try to convert to `float`. If conversion fails → `(False, "invalid_price")`.
3. If the number is **< 0** → `(False, "negative_price")`.
4. Else → `(True, None)`.

**Hint:** the structure often looks like:

```text
if  missing:
    return (False, "missing_price")
try:
    price = float(...)
except ValueError:
    return (False, "invalid_price")
...
```

**Stretch:** add one `assert` for a good price and one for a bad price.

In [ ]:
def apply_price_rule(price_str):
    raise NotImplementedError("Your turn: implement apply_price_rule")

# assert apply_price_rule("10") == (True, None)
# assert apply_price_rule("") == (False, "missing_price")

---

## Challenge C — Many rows: count and find duplicates

**Story:** you received many rows. Some `product_id` values might be **empty** (bad). Some might appear **twice** (duplicates).

**Given:** `rows`, a **list** of dictionaries. Each dict has at least the key `"product_id"`.

**Return two things:**

1. **`good_count`:** how many rows have a **non-empty** `product_id` **after** strip.
2. **`duplicate_ids`:** a **set** of `product_id` strings that appear **more than once** (count only among rows where `product_id` strip is not empty).

### Hints

- A **`set`** stores unique items — good for “have I seen this id before?”
- You might also use a **dictionary** to count how many times each id appeared — both styles are fine.
- Walk through a small list on paper first (3–4 rows).

**Function name:** `analyze_rows(rows)` → return `(good_count, duplicate_ids)`.

In [ ]:
def analyze_rows(rows):
    raise NotImplementedError("Your turn: implement analyze_rows")

# sample = [
#     {"product_id": "  A "},
#     {"product_id": "A"},
#     {"product_id": ""},
# ]
# good_count, dups = analyze_rows(sample)
# print(good_count, dups)

---

## What comes next

Topic 2 introduces **lists, sets, and comprehensions** with the same style: explanation first, small examples, then challenges. Take a break before the next notebook if this was a lot — that is normal.

**Well done** for working through the basics; pipelines are built from small, clear steps like the ones here.